# 04. 무패턴 진단 — ① 돈의 출처(상류) · ② 같은 분 같은 쌍

학습 없이 SQL 집계만 한다. 새 피처를 만들기 전에 **신호가 있는지, 있으면 얼마나 순도 높은 구간을 만드는지** 먼저 본다.

| 진단 | 질문 | 계산 |
|---|---|---|
| ① 상류 출처 | 송신 계좌가 이 거래 **전에** 세탁 거래로 돈을 받았나? 얼마나 최근에? | 송신 계좌의 마지막 패턴 양성 입금 / 무패턴 양성 입금 시각 → 경과 시간 구간 |
| ② 같은 분 같은 쌍 | 처음(또는 거의 처음) 보는 쌍이 같은 분에 또 보내면 얼마나 세탁인가? | 쌍의 과거 건수(같은 분 제외) × 같은 분 같은 쌍 다른 거래 수 |

- 라벨: `cls` 0 정상 · 1 패턴 양성(`patterns.parquet` 연결) · 2 무패턴 양성
- 이력: 2022-08-01 ~ 10-30 전체 거래(패턴 포함). 집계 대상: 우리 분할 Train 8/8~10/2 · Val 10/3~10/16 · Test 10/17~10/30
- **같은 분은 '과거'로 안 센다** (분 단위 데이터라 같은 분 안의 순서를 모름). ①은 `입금 시각 < 거래 시각`
- ①은 진단용으로 **라벨을 직접 쓴다**. 피처로 쓸 때는 라벨 대신 1단계 모델 점수(전날까지)로 바꿔야 한다
- 집중 송신 집단(`hot`): 팀원 문서의 "은행 70 송신 계좌 15개" — 무패턴을 많이 보낸 송신 계좌. 이 집단은 거래·계좌 피처로 정상과 구분이 안 된다고 보고됐으므로 **따로 나눠 본다**

## 0. 설정

GPU 서버 Jupyter, `code/nonpaten_vs_normal` 에서 위에서부터 실행. 필요한 입력은 `code/01_prepare_parquet.ipynb` 출력 세 개다.

```text
parquets/HI-Large/transactions.parquet
parquets/HI-Large/patterns.parquet
parquets/HI-Large/kyc.parquet
```

중간 테이블은 `parquets/HI-Large/nonpaten_vs_normal/04_diagnosis/work.duckdb` 에 쌓는다. 커널이 죽어도 다시 돌리면 **이미 만든 단계는 건너뛴다** (`REBUILD=True` 면 새로 만듦).

In [ ]:
# 현재 Jupyter 커널에 패키지가 없을 때만 주석을 해제해 실행합니다.
# !pip install --break-system-packages --no-cache-dir -q duckdb pandas

In [ ]:
from pathlib import Path
import json, time
import duckdb, numpy as np, pandas as pd
from IPython.display import display

DATASET = "HI-Large"
PROJECT_ROOT = None            # 자동 탐색 실패 시 Path("/workspace/money_laundry_ai_")
THREADS, MEMORY = 4, "8GB"     # DuckDB (넘치면 work 폴더 tmp/ 로 내려 씀)
REBUILD = False                # True: 중간 테이블 새로 만들기

HISTORY = ("2022-08-01", "2022-10-31")   # 이력으로 쓰는 거래 [시작, 끝) — 팀원 분석 기간과 같음
SPLITS = {"train": ("2022-08-08", "2022-10-02"),     # 끝 날짜 포함
          "val":   ("2022-10-03", "2022-10-16"),
          "test":  ("2022-10-17", "2022-10-30")}
HOT_MIN = 300                  # 이력 기간에 무패턴을 이 건수 이상 보낸 송신 계좌 = 집중 송신 집단
TEAM = {"nonpattern": 82_260, "hot_accounts": 15, "hot_nonpattern": 22_979}   # 팀원 문서 수치 (비교용)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)


def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "parquets" / DATASET / "transactions.parquet").is_file():
            return c
    raise FileNotFoundError("transactions.parquet 를 찾지 못했습니다. PROJECT_ROOT 를 지정하세요.")


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE = ROOT / "parquets" / DATASET
TX, PAT, KYC = BASE / "transactions.parquet", BASE / "patterns.parquet", BASE / "kyc.parquet"
OUT = BASE / "nonpaten_vs_normal" / "04_diagnosis"
for p in (TX, PAT, KYC):
    assert p.is_file(), p
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT :", OUT)

In [ ]:
# DuckDB 연결 (파일 DB — 큰 중간 테이블을 디스크에 둔다)
con = duckdb.connect(str(OUT / "work.duckdb"))
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{MEMORY}'")
con.execute(f"SET temp_directory='{OUT / 'tmp'}'")
con.execute("SET preserve_insertion_order=false")


def q(v):
    return "'" + str(v).replace("'", "''") + "'"


def has(name):
    return con.execute("SELECT count(*) FROM duckdb_tables() WHERE table_name = ?", [name]).fetchone()[0] > 0


def step(name, sql):
    """CREATE TABLE name AS sql — 이미 있으면 건너뜀"""
    if has(name) and not REBUILD:
        print(f"{name:12s} 이미 있음 — 건너뜀 ({con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]:,} 행)")
        return
    t = time.perf_counter()
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql}")
    n = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
    print(f"{name:12s} {n:>13,} 행  ({time.perf_counter() - t:,.1f}s)", flush=True)


def split_sql(col="ts"):
    """분할 이름 (분할 밖이면 NULL)"""
    return "CASE " + " ".join(
        f"WHEN {col} >= TIMESTAMP {q(a)} AND {col} < DATE {q(b)} + INTERVAL 1 DAY THEN {q(s)}" for s, (a, b) in SPLITS.items()
    ) + " END"


print(split_sql())

## 1. 거래 + 라벨 `tx`

필요한 열만 남긴다: `ts, src, dst, cls`. 패턴 연결은 01 노트북과 같이 `edge_id` 로 한다.

In [ ]:
step("tx", f"""
    SELECT t.ts, t.src, t.dst,
           (CASE WHEN NOT t.is_laundering THEN 0 WHEN p.edge_id IS NOT NULL THEN 1 ELSE 2 END)::TINYINT AS cls
    FROM read_parquet({q(TX)}) t
    LEFT JOIN (SELECT DISTINCT edge_id FROM read_parquet({q(PAT)}) WHERE NOT unmatched) p ON p.edge_id = t.edge_id
    WHERE t.ts >= TIMESTAMP {q(HISTORY[0])} AND t.ts < TIMESTAMP {q(HISTORY[1])}
""")
# 패턴에 연결됐는데 정상 라벨인 거래가 없어야 한다
bad = con.execute(f"""SELECT count(*) FROM read_parquet({q(TX)}) t JOIN read_parquet({q(PAT)}) p ON p.edge_id = t.edge_id
                      WHERE NOT p.unmatched AND NOT t.is_laundering""").fetchone()[0]
assert bad == 0, f"패턴 연결 정상 거래 {bad:,}"

In [ ]:
# 구간 × 라벨 건수 (이력만 = 8/1~8/7, 10/31 은 집계 대상 아님)
counts = con.execute(f"""
    SELECT coalesce({split_sql()}, '(이력만)') AS split,
           count(*) FILTER (WHERE cls = 0) AS normal, count(*) FILTER (WHERE cls = 1) AS pattern,
           count(*) FILTER (WHERE cls = 2) AS nonpattern, count(*) AS rows
    FROM tx GROUP BY 1 ORDER BY min(ts)
""").df().set_index("split")
counts.loc["합계"] = counts.sum()
counts["nonpattern_rate_%"] = (100 * counts.nonpattern / counts.rows).round(4)
display(counts)
print(f"이력 기간 무패턴 {int(counts.loc['합계','nonpattern']):,}  (팀원 문서 {TEAM['nonpattern']:,})")

## 2. 집중 송신 집단 `hot`

송신 계좌별 무패턴 건수. 팀원 문서: 은행 70 의 송신 계좌 15개가 무패턴 22,979건(약 28%)을 보냈고, 같은 계좌들이 정상 거래를 1,620만 건 보냈다.
분포를 보고 `HOT_MIN` 이 그 집단을 정확히 가르는지 확인한다.

In [ ]:
step("sender_np", f"""
    SELECT t.src, k.bank_id, k.bank_name, k.entity_id, k.entity_name,
           count(*) FILTER (WHERE cls = 2) AS nonpattern, count(*) FILTER (WHERE cls = 1) AS pattern,
           count(*) FILTER (WHERE cls = 0) AS normal
    FROM tx t JOIN read_parquet({q(KYC)}) k ON k.node_id = t.src
    GROUP BY ALL HAVING count(*) FILTER (WHERE cls = 2) > 0
""")
# 송신 계좌 수 분포 (무패턴 건수 구간별)
display(con.execute("""
    SELECT CASE WHEN nonpattern = 1 THEN '1' WHEN nonpattern < 10 THEN '2~9' WHEN nonpattern < 100 THEN '10~99'
                WHEN nonpattern < 1000 THEN '100~999' ELSE '1000+' END AS nonpattern_per_sender,
           count(*) AS senders, sum(nonpattern) AS nonpattern, sum(normal) AS normal
    FROM sender_np GROUP BY 1 ORDER BY min(nonpattern)
""").df())
con.execute("SELECT * FROM sender_np ORDER BY nonpattern DESC LIMIT 30").df()

In [ ]:
# HOT_MIN 을 바꾸면 바로 반영되도록 매번 새로 만든다
con.execute(f"CREATE OR REPLACE TABLE hot AS SELECT src FROM sender_np WHERE nonpattern >= {HOT_MIN}")
h = con.execute("""SELECT count(*) AS accounts, sum(nonpattern) AS nonpattern, sum(normal) AS normal, sum(pattern) AS pattern,
                          count(DISTINCT bank_id) AS banks, list(DISTINCT bank_id) AS bank_ids, count(DISTINCT entity_id) AS entities
                   FROM sender_np JOIN hot USING (src)""").df()
display(h)
print(f"팀원 문서: 계좌 {TEAM['hot_accounts']}개, 무패턴 {TEAM['hot_nonpattern']:,}건 — 다르면 HOT_MIN 을 조정")

## 3. ① 상류 출처 — 송신 계좌가 마지막으로 세탁 입금을 받은 뒤 얼마나 지났나

`(src, ts)` 마다 `src` 가 **받은** 패턴 양성 거래 / 무패턴 양성 거래 중 `ts` 보다 이전 분의 마지막 시각을 ASOF JOIN 으로 붙인다.

경과 구간: `≤1시간` · `≤1일` · `≤7일` · `≤30일` · `>30일` · `없음`(이력 시작 8/1 이후 받은 적 없음)

- **패턴 입금**: 피처로 바꿀 수 있는 쪽 (1단계 모델이 패턴은 거의 다 잡으므로 점수로 대체 가능)
- **무패턴 입금**: 라벨 생성 방식 확인용 (무패턴이 무패턴을 낳는가). 피처로는 못 씀

In [ ]:
# 세탁 입금 시각 (받은 계좌 기준, 분 단위 중복 제거)
step("in_pat", "SELECT DISTINCT dst AS acct, ts FROM tx WHERE cls = 1")
step("in_np",  "SELECT DISTINCT dst AS acct, ts FROM tx WHERE cls = 2")
# 집계 대상 분할의 (송신 계좌, 분)
step("src_ts", f"SELECT DISTINCT src, ts FROM tx WHERE ({split_sql()}) IS NOT NULL")
# 직전(이전 분) 세탁 입금 시각
step("up", """
    SELECT s.src, s.ts, p.ts AS last_pat_in, r.ts AS last_np_in
    FROM src_ts s
    ASOF LEFT JOIN in_pat p ON s.src = p.acct AND s.ts > p.ts
    ASOF LEFT JOIN in_np  r ON s.src = r.acct AND s.ts > r.ts
""")
con.execute("SELECT * FROM up WHERE last_pat_in IS NOT NULL LIMIT 5").df()

## 4. ② 같은 분 같은 쌍 — 쌍 과거 건수 × 같은 분 다른 거래 수

`(src, dst, 분)` 칸으로 묶은 뒤 쌍별 누적합으로 **같은 분 이전까지의 쌍 건수**(`prior_n`)를 구한다.
칸 안의 거래 수가 `n` 이면 각 거래의 같은 분 같은 쌍 다른 거래 수는 `n − 1` 이다. 이력 8/1 부터, 패턴 거래 포함.

In [ ]:
step("pair_cells", """
    SELECT src, dst, ts, count(*)::INT AS n,
           count(*) FILTER (WHERE cls = 0)::INT AS n0, count(*) FILTER (WHERE cls = 1)::INT AS n1,
           count(*) FILTER (WHERE cls = 2)::INT AS n2
    FROM tx GROUP BY ALL
""")
step("pair_cum", """
    SELECT *, coalesce(sum(n) OVER (PARTITION BY src, dst ORDER BY ts
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0)::BIGINT AS prior_n
    FROM pair_cells
""")
con.execute("SELECT * FROM pair_cum WHERE n > 1 ORDER BY src, dst, ts LIMIT 8").df()

## 5. 한 표로 모으기 `agg`

분할 × hot × ①패턴 입금 구간 × ①무패턴 입금 구간 × ②과거 건수 구간 × ②같은 분 구간 → 라벨별 건수.
이후 표는 모두 이 작은 표에서 pandas 로 만든다.

In [ ]:
GAP = ["≤1시간", "≤1일", "≤7일", "≤30일", ">30일", "없음"]
PRIOR = ["0", "1", "2~5", "6~20", "21+"]
SAME = ["0", "1", "2~4", "5+"]


def gap_code(col):
    m = f"date_diff('minute', {col}, c.ts)"
    return (f"CASE WHEN {col} IS NULL THEN 5 WHEN {m} <= 60 THEN 0 WHEN {m} <= 1440 THEN 1 "
            f"WHEN {m} <= 10080 THEN 2 WHEN {m} <= 43200 THEN 3 ELSE 4 END")


agg = con.execute(f"""
    SELECT {split_sql('c.ts')} AS split, (h.src IS NOT NULL) AS hot,
           {gap_code('u.last_pat_in')} AS g_pat, {gap_code('u.last_np_in')} AS g_np,
           CASE WHEN prior_n = 0 THEN 0 WHEN prior_n = 1 THEN 1 WHEN prior_n <= 5 THEN 2 WHEN prior_n <= 20 THEN 3 ELSE 4 END AS prior_b,
           CASE WHEN n = 1 THEN 0 WHEN n = 2 THEN 1 WHEN n <= 5 THEN 2 ELSE 3 END AS same_b,
           sum(n0) AS n0, sum(n1) AS n1, sum(n2) AS n2
    FROM pair_cum c
    JOIN up u ON u.src = c.src AND u.ts = c.ts
    LEFT JOIN hot h ON h.src = c.src
    GROUP BY ALL
""").df()
agg["split"] = pd.Categorical(agg.split, list(SPLITS))
for col, labels in [("g_pat", GAP), ("g_np", GAP), ("prior_b", PRIOR), ("same_b", SAME)]:
    agg[col] = pd.Categorical.from_codes(agg[col].astype(int), labels)
agg.to_parquet(OUT / "agg.parquet", index=False)

# 분할별 건수가 1번 표와 같아야 한다 (빠진 거래 없음)
chk = agg.groupby("split", observed=True)[["n0", "n1", "n2"]].sum()
assert (chk.values == counts.loc[list(SPLITS), ["normal", "pattern", "nonpattern"]].values).all(), chk
print(len(agg), "칸 · 분할별 건수 1번 표와 일치")

In [ ]:
def table(df, by):
    """구간별 건수와 순도.
    무패턴_비율 = 무패턴 / (정상 + 무패턴)  (패턴 제외, 03 과 같은 기준)
    양성_비율   = (패턴 + 무패턴) / 전체     (이진 모델 운영점 기준)
    무패턴_비중 / 정상_비중 = 이 구간에 들어간 무패턴·정상의 비율, 배수 = 둘의 비"""
    t = df.groupby(by, observed=True)[["n0", "n1", "n2"]].sum()
    t["rows"] = t.n0 + t.n1 + t.n2
    t["무패턴_비율_%"] = 100 * t.n2 / (t.n0 + t.n2).clip(lower=1)
    t["양성_비율_%"] = 100 * (t.n1 + t.n2) / t.rows.clip(lower=1)
    t["무패턴_비중_%"] = 100 * t.n2 / max(t.n2.sum(), 1)
    t["정상_비중_%"] = 100 * t.n0 / max(t.n0.sum(), 1)
    t["배수"] = t["무패턴_비중_%"] / t["정상_비중_%"].replace(0, np.nan)
    return t.round(3)


def rule_curve(df, by, top=12):
    """구간을 양성 비율 높은 순으로 하나씩 더해 갈 때 P / 전체 R / 무패턴 R — 이 피처 하나만으로 만든 규칙의 상한"""
    t = df.groupby(by, observed=True)[["n0", "n1", "n2"]].sum()
    t = t[(t.n0 + t.n1 + t.n2) > 0]
    t["rows"] = t.n0 + t.n1 + t.n2
    t["pos"] = t.n1 + t.n2
    t = t.assign(rate=t.pos / t.rows).sort_values("rate", ascending=False).drop(columns="rate")
    cum = t.cumsum()
    t["P_%"] = 100 * cum.pos / cum.rows
    t["R_전체_%"] = 100 * cum.pos / t.pos.sum()
    t["R_무패턴_%"] = 100 * cum.n2 / max(t.n2.sum(), 1)
    return t.head(top).round(3)

## 6. ① 결과 — 상류 출처

먼저 전체 분할 합계를 hot / 나머지로 나눠 본다. 읽는 법:
- `무패턴_비중` 과 `정상_비중` 이 크게 다르면(배수 ≫ 1) 신호가 있다
- 그 구간의 `무패턴_비율` 이 기준선(1번 표 약 0.05%)보다 몇 배인지가 순도

In [ ]:
# 패턴 양성 입금 기준 (피처로 바꿀 수 있는 쪽)
for hot, name in [(True, "hot 집단"), (False, "나머지")]:
    print(f"── {name} · 송신 계좌의 직전 패턴 양성 입금")
    display(table(agg[agg.hot == hot], "g_pat"))

In [ ]:
# 무패턴 양성 입금 기준 (라벨 생성 확인용)
for hot, name in [(True, "hot 집단"), (False, "나머지")]:
    print(f"── {name} · 송신 계좌의 직전 무패턴 양성 입금")
    display(table(agg[agg.hot == hot], "g_np"))

In [ ]:
# hot 계좌들은 세탁 입금을 받는가 — 계좌별 받은 패턴·무패턴 양성 건수
con.execute("""
    WITH recv AS (SELECT dst AS src, count(*) FILTER (WHERE cls = 1) AS recv_pattern,
                         count(*) FILTER (WHERE cls = 2) AS recv_nonpattern, count(*) AS recv_all
                  FROM tx WHERE dst IN (SELECT src FROM hot) GROUP BY dst)
    SELECT s.src, s.bank_id, s.nonpattern AS sent_nonpattern, s.normal AS sent_normal,
           coalesce(r.recv_pattern, 0) AS recv_pattern, coalesce(r.recv_nonpattern, 0) AS recv_nonpattern,
           coalesce(r.recv_all, 0) AS recv_all
    FROM sender_np s JOIN hot USING (src) LEFT JOIN recv r USING (src) ORDER BY sent_nonpattern DESC
""").df()

In [ ]:
# 분할별로 같은 경향인가 (나머지 집단, 패턴 입금 기준)
rest = agg[~agg.hot]
pd.concat({s: table(rest[rest.split == s], "g_pat")[["n2", "무패턴_비율_%", "무패턴_비중_%", "정상_비중_%", "배수"]]
           for s in SPLITS}, axis=1)

In [ ]:
# 규칙 상한: 패턴 입금 구간 하나만으로 (분할 합계, 전체 집단)
rule_curve(agg, "g_pat")

## 7. ② 결과 — 같은 분 같은 쌍

03 결과(무패턴 데이터): 과거 0건 쌍이 같은 분에 또 보내면 무패턴 비율 17.9%, 과거 1건이면 49.1%.
여기서는 **패턴 거래도 포함한 전체 원장**이라 같은 분 다른 거래에 패턴 거래도 센다.

In [ ]:
t = table(agg, ["prior_b", "same_b"])
print("무패턴 비율 % (패턴 제외) — 행: 쌍 과거 건수, 열: 같은 분 같은 쌍 다른 거래 수")
display(t["무패턴_비율_%"].unstack())
print("무패턴 건수")
display(t["n2"].unstack())
print("양성 비율 % (패턴 포함)")
display(t["양성_비율_%"].unstack())

In [ ]:
# hot / 나머지
pd.concat({name: table(agg[agg.hot == hot], ["prior_b", "same_b"])[["n2", "무패턴_비율_%", "무패턴_비중_%"]]
           for hot, name in [(True, "hot"), (False, "나머지")]}, axis=1).query("same_b != '0'")

In [ ]:
# 분할별 안정성 — 같은 분 1건 이상 칸만
pd.concat({s: table(agg[agg.split == s], ["prior_b", "same_b"])[["n2", "무패턴_비율_%", "양성_비율_%"]]
           for s in SPLITS}, axis=1).query("same_b != '0'")

In [ ]:
# 규칙 상한: (과거 건수, 같은 분) 칸만으로
rule_curve(agg, ["prior_b", "same_b"])

## 8. ① × ② 겹침 — 두 신호가 같은 무패턴을 가리키나

무패턴 거래(분할 합계)를 "직전 7일 안에 패턴 양성 입금" × "같은 분 같은 쌍 1건 이상" 으로 나눈다. 많이 안 겹치면 둘 다 넣을 가치가 있다.

In [ ]:
x = agg.assign(up7=agg.g_pat.isin(["≤1시간", "≤1일", "≤7일"]), same=agg.same_b != "0")
pd.concat({name: x[x.hot == hot].groupby(["up7", "same"])[["n0", "n2"]].sum().assign(
               무패턴_비율_pct=lambda d: (100 * d.n2 / (d.n0 + d.n2)).round(3),
               무패턴_비중_pct=lambda d: (100 * d.n2 / d.n2.sum()).round(2))
           for hot, name in [(True, "hot"), (False, "나머지")]}, axis=1)

## 9. 요약 저장 · 판단 기준

| 결과 | 다음 단계 |
|---|---|
| ① 나머지 집단에서 `≤1일`/`≤7일` 배수가 크고(예: 5배 이상) 무패턴 비중도 수 % 이상 | 05 에서 "직전 k일 고위험 입금" 피처로 만든다 (1단계 점수 사용) |
| ① hot 집단에서 무패턴만 세탁 입금 직후에 몰림 | hot 집단을 가를 유일한 단서 → 우선순위 1 |
| ① 배수 ≈ 1 | ① 접음 |
| ② 같은 분 칸 순도가 분할마다 비슷하게 유지 | 05 에 `pair_same_minute_other_n` 등 추가 |

In [ ]:
summary = {}
for hot, name in [(True, "hot"), (False, "rest"), (None, "all")]:
    d = agg if hot is None else agg[agg.hot == hot]
    t = table(d, "g_pat").reindex(GAP).fillna(0)
    near = ["≤1시간", "≤1일", "≤7일"]
    summary[name] = {
        "nonpattern": int(d.n2.sum()), "normal": int(d.n0.sum()),
        "pat_in_7d_nonpattern_%": round(float(t.loc[near, "무패턴_비중_%"].sum()), 3),
        "pat_in_7d_normal_%": round(float(t.loc[near, "정상_비중_%"].sum()), 3),
        "same_minute_nonpattern_%": round(float(100 * d.loc[d.same_b != "0", "n2"].sum() / max(d.n2.sum(), 1)), 3),
        "same_minute_normal_%": round(float(100 * d.loc[d.same_b != "0", "n0"].sum() / max(d.n0.sum(), 1)), 3),
    }
summary = {"history": HISTORY, "splits": SPLITS, "hot_min": HOT_MIN, **summary}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
table(agg, "g_pat").to_csv(OUT / "upstream_pattern_all.csv")
table(agg, ["prior_b", "same_b"]).to_csv(OUT / "same_minute_pair_all.csv")
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# 정리: 중간 테이블은 work.duckdb 에 남겨 둔다 (다시 실행 시 재사용). 지우려면 DELETE_WORK=True
DELETE_WORK = False
con.close()
if DELETE_WORK:
    (OUT / "work.duckdb").unlink(missing_ok=True)
    (OUT / "work.duckdb.wal").unlink(missing_ok=True)
print("완료. 결과:", OUT)